<a href="https://colab.research.google.com/github/nyp-sit/iti121-2025s2/blob/main/L11/finetune_embedding_llamaindex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetune Embeddings


Finetuning embedding models often heavily improves the performance of the model on your use case, because each task requires a different notion of similarity. For example, given news articles:
- “Apple launches the new iPad”
- “NVIDIA is gearing up for the next GPU generation”
Then the following use cases, we may have different notions of similarity:
- a model for classification of news articles as Economy, Sports, Technology, Politics, etc., should produce similar embeddings for these texts.
- a model for semantic textual similarity should produce dissimilar embeddings for these texts, as they have different meanings.
- a model for semantic search would not need a notion for similarity between two documents, as it should only compare queries and documents.

In this notebook, we show users how to easily finetune their own embedding models using LLamaIndex.

We go through three main sections:
1. Preparing the data (our `generate_qa_embedding_pairs` function makes this easy)
2. Finetuning the model (using our `SentenceTransformersFinetuneEngine`)
3. Evaluating the model on a validation knowledge corpus

## Generate Corpus

First, we create the corpus of text chunks by leveraging LlamaIndex to load some financial PDFs, and parsing/chunking into plain text chunks.

In [ ]:
# %pip install datasets
%pip -q install llama-index-llms-openai
%pip -q install llama-index-embeddings-openai
%pip -q install llama-index-llms-azure-openai
%pip -q install llama-index-finetuning
%pip -q install llama-index-readers-file
%pip -q install llama-index-embeddings-huggingface
%pip -q install openai

# also need to import the batch_util functions
!wget https://raw.githubusercontent.com/nyp-sit/iti121-2025s2/refs/heads/main/L11/batch_generate.py

In [ ]:
import json
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode

Now let's us download two PDFs (annual report of Uber and Lyft filed with US Securities and Exchange Commission), and use them to generate some queries and answer pairs.

In [ ]:
!mkdir -p 'data/10k/'
!wget -q 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/10k/uber_2021.pdf' -O 'data/10k/uber_2021.pdf'
!wget -q 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/10k/lyft_2021.pdf' -O 'data/10k/lyft_2021.pdf'

In [ ]:
TRAIN_FILES = ["./data/10k/lyft_2021.pdf"]
VAL_FILES = ["./data/10k/uber_2021.pdf"]

TRAIN_CORPUS_FPATH = "./data/train_corpus.json"
VAL_CORPUS_FPATH = "./data/val_corpus.json"

We will use Simple

In [ ]:
def load_corpus(files, verbose=False):
    if verbose:
        print(f"Loading files {files}")

    reader = SimpleDirectoryReader(input_files=files)
    docs = reader.load_data()
    if verbose:
        print(f"Loaded {len(docs)} docs")

    parser = SentenceSplitter()
    nodes = parser.get_nodes_from_documents(docs, show_progress=verbose)

    if verbose:
        print(f"Parsed {len(nodes)} nodes")

    return nodes

We do a very naive train/val split by having the Lyft corpus as the train dataset, and the Uber corpus as the val dataset.

In [ ]:
train_nodes = load_corpus(TRAIN_FILES, verbose=True)
val_nodes = load_corpus(VAL_FILES, verbose=True)

### Generate synthetic queries

Now, we use an LLM (gpt-3.5-turbo) to generate questions using each text chunk in the corpus as context.

Each pair of (generated question, text chunk used as context) becomes a datapoint in the finetuning dataset (either for training or evaluation).

In [ ]:
from batch_generate import generate_qa_embedding_pairs_batch
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

In [ ]:
from openai import OpenAI

endpoint = "https://nypopenai2.cognitiveservices.azure.com/openai/v1/"
model_name = "gpt-4.1-nano"
deployment_name = "gpt-4.1-nano"

api_key = "<<apikey>>"

client = OpenAI(
    base_url=f"{endpoint}",
    api_key=api_key
)


The `generate_qa_embedding_pairs_batch()` is a async function.  We will call it twice, one to generate train_dataset, and one to generate validation_dataset.  We will wait for both to complete.

In [ ]:
import asyncio


results = await asyncio.gather(
    generate_qa_embedding_pairs_batch(
        nodes=train_nodes,
        openai_client=client,
        model="gpt-4.1-nano",
        num_questions_per_chunk=2,
        output_path="train_dataset.json"),
    generate_qa_embedding_pairs_batch(
        nodes=val_nodes,
        openai_client=client,
        model="gpt-4.1-nano",
        num_questions_per_chunk=2,
        output_path="val_dataset.json")
)

In [ ]:
train_dataset = results[0]
val_dataset = results[1]

In [ ]:
# [Optional] Load
train_dataset1 = EmbeddingQAFinetuneDataset.from_json("train_dataset.json")
val_dataset2 = EmbeddingQAFinetuneDataset.from_json("val_dataset.json")

## Run Embedding Finetuning

In [ ]:
from llama_index.finetuning import SentenceTransformersFinetuneEngine

In [ ]:
finetune_engine = SentenceTransformersFinetuneEngine(
    train_dataset,
    model_id="BAAI/bge-small-en",
    model_output_path="test_model",
    val_dataset=val_dataset,
)

In [ ]:
finetune_engine.finetune()

In [ ]:
embed_model = finetune_engine.get_finetuned_model()

## Evaluate Finetuned Model

In this section, we evaluate 3 different embedding models:
1. proprietary OpenAI embedding,
2. open source `BAAI/bge-small-en`, and
3. our finetuned embedding model.

We consider 2 evaluation approaches:
1. a simple custom **hit rate** metric
2. using `InformationRetrievalEvaluator` from sentence_transformers

We show that finetuning on synthetic (LLM-generated) dataset significantly improve upon an opensource embedding model.

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import TextNode
from tqdm.notebook import tqdm
import pandas as pd

### Define eval function

**Option 1**: We use a simple **hit rate** metric for evaluation:
* for each (query, relevant_doc) pair,
* we retrieve top-k documents with the query,  and
* it's a **hit** if the results contain the relevant_doc.

This approach is very simple and intuitive, and we can apply it to both the proprietary OpenAI embedding as well as our open source and fine-tuned embedding models.

In [ ]:
def evaluate(
    dataset,
    embed_model,
    top_k=5,
    verbose=False,
):
    corpus = dataset.corpus
    queries = dataset.queries
    relevant_docs = dataset.relevant_docs

    nodes = [TextNode(id_=id_, text=text) for id_, text in corpus.items()]
    index = VectorStoreIndex(
        nodes, embed_model=embed_model, show_progress=True
    )
    retriever = index.as_retriever(similarity_top_k=top_k)

    eval_results = []
    for query_id, query in tqdm(queries.items()):
        retrieved_nodes = retriever.retrieve(query)
        retrieved_ids = [node.node.node_id for node in retrieved_nodes]
        expected_id = relevant_docs[query_id][0]
        is_hit = expected_id in retrieved_ids  # assume 1 relevant doc

        eval_result = {
            "is_hit": is_hit,
            "retrieved": retrieved_ids,
            "expected": expected_id,
            "query": query_id,
        }
        eval_results.append(eval_result)
    return eval_results

**Option 2**: We use the `InformationRetrievalEvaluator` from sentence_transformers.

This provides a more comprehensive suite of metrics, but we can only run it against the sentencetransformers compatible models (open source and our finetuned model, *not* the OpenAI embedding model).

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import SentenceTransformer
from pathlib import Path


def evaluate_st(
    dataset,
    model_id,
    name,
):
    corpus = dataset.corpus
    queries = dataset.queries
    relevant_docs = dataset.relevant_docs

    evaluator = InformationRetrievalEvaluator(
        queries, corpus, relevant_docs, name=name
    )
    model = SentenceTransformer(model_id)
    output_path = "results/"
    Path(output_path).mkdir(exist_ok=True, parents=True)
    return evaluator(model, output_path=output_path)

### Run Evals

#### OpenAI

Note: this might take a few minutes to run since we have to embed the corpus and queries

In [ ]:
!pip install llama-index-embeddings-azure-openai

In [ ]:
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

azure_openai_embed = AzureOpenAIEmbedding(
        model="text-embedding-ada-002",  # Or your specific embedding model
        deployment_name="text-embedding-ada-002"
    )

In [ ]:
# ada = OpenAIEmbedding()
ada = azure_openai_embed
ada_val_results = evaluate(val_dataset, ada)

In [ ]:
df_ada = pd.DataFrame(ada_val_results)

In [ ]:
hit_rate_ada = df_ada["is_hit"].mean()
hit_rate_ada

### BAAI/bge-small-en

In [ ]:
bge = "local:BAAI/bge-small-en"
bge_val_results = evaluate(val_dataset, bge)

In [ ]:
df_bge = pd.DataFrame(bge_val_results)

In [ ]:
hit_rate_bge = df_bge["is_hit"].mean()
hit_rate_bge

In [ ]:
evaluate_st(val_dataset, "BAAI/bge-small-en", name="bge")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Finetuned

In [ ]:
finetuned = "local:test_model"
val_results_finetuned = evaluate(val_dataset, finetuned)

In [ ]:
df_finetuned = pd.DataFrame(val_results_finetuned)

In [ ]:
hit_rate_finetuned = df_finetuned["is_hit"].mean()
hit_rate_finetuned

In [ ]:
evaluate_st(val_dataset, "test_model", name="finetuned")

### Summary of Results

#### Hit rate

In [ ]:
df_ada["model"] = "ada"
df_bge["model"] = "bge"
df_finetuned["model"] = "fine_tuned"

We can see that fine-tuning our small open-source embedding model drastically improve its retrieval quality (even approaching the quality of the proprietary OpenAI embedding)!

In [ ]:
df_all = pd.concat([df_ada, df_bge, df_finetuned])
df_all.groupby("model").mean("is_hit")

#### InformationRetrievalEvaluator

In [ ]:
df_st_bge = pd.read_csv(
    "results/Information-Retrieval_evaluation_bge_results.csv"
)
df_st_finetuned = pd.read_csv(
    "results/Information-Retrieval_evaluation_finetuned_results.csv"
)

We can see that embedding finetuning improves metrics consistently across the suite of eval metrics

In [ ]:
df_st_bge["model"] = "bge"
df_st_finetuned["model"] = "fine_tuned"
df_st_all = pd.concat([df_st_bge, df_st_finetuned])
df_st_all = df_st_all.set_index("model")
df_st_all